# Carbon Transition DuckDB Workflow

This notebook demonstrates the local DuckDB workflow using synthetic OWID-like data. Run the same commands with real OWID files after `carbon-duckdb download-owid`.

## 1. Generate synthetic raw data
The sample data keeps the workflow reproducible without network access.

In [ ]:
from pathlib import Path
from carbon_transition_duckdb.sample_data import generate_synthetic_owid_data

raw_dir = Path('../data/raw')
generate_synthetic_owid_data(raw_dir, start_year=2010, end_year=2024)


## 2. Build the DuckDB lakehouse
The database is embedded in a single `.duckdb` file.

In [ ]:
from carbon_transition_duckdb.config import ProjectPaths
from carbon_transition_duckdb.pipeline import build_duckdb_lakehouse

paths = ProjectPaths(
    raw_dir=raw_dir,
    database=Path('../data/processed/carbon_transition.duckdb'),
    export_dir=Path('../data/processed/marts'),
)
build_duckdb_lakehouse(paths)


## 3. Score transition risk
Scores are transparent and component-based.

In [ ]:
from carbon_transition_duckdb.pipeline import compute_transition_scores

scores = compute_transition_scores(paths.database)
scores.sort_values('transition_risk_score', ascending=False).head(10)


## 4. Generate a report
The report is a compact Markdown artifact for review.

In [ ]:
from carbon_transition_duckdb.reporting.markdown import write_report

report_path = Path('../reports/sample_run/transition_report.md')
write_report(scores, report_path, title='Synthetic Carbon Transition Report')
report_path
